# Gráficas de carpetas y datos faltantes — corte vigente

Este notebook genera sólo dos gráficas de barras con datos del perfil activo `reporte_diario` y sus insumos ETL. Las cifras se recalculan al ejecutar el notebook; no se transcriben de imágenes.

La primera gráfica sigue las categorías del reporte diario: desaparición forzada o cometida por particulares y carpetas válidas de otros delitos. La segunda muestra sólo la gráfica de barras, sin tarjetas ni elementos de diseño de una diapositiva. «Nombre faltante» significa que los tres componentes del nombre están vacíos. Los valores literales como `N/A` se conservan como texto, conforme al ETL del proyecto; sólo nulos y cadenas vacías cuentan como faltantes.

In [ ]:

from pathlib import Path
import sys
import unicodedata

RAIZ_PROYECTO = next(
    (ruta for ruta in (Path.cwd(), *Path.cwd().parents)
     if (ruta / "src").is_dir() and (ruta / "config.py").is_file()),
    None,
)
if RAIZ_PROYECTO is None:
    raise RuntimeError("No se encontró la raíz del proyecto (se esperaban src/ y config.py en un directorio padre).")
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib import font_manager
from matplotlib.ticker import FuncFormatter

from src.settings import Settings
from src.pipeline.universo import Universo
from src.slices import Slice
from src.cortes_referencia import bajas_vs_corte

for archivo_fuente in ("Montserrat-Regular.ttf", "Montserrat-Bold.ttf"):
    font_manager.fontManager.addfont(RAIZ_PROYECTO / "0_fonts" / archivo_fuente)

plt.rcParams.update({
    "font.family": "Montserrat",
    "font.size": 13,
})

VINO = "#721c37"
GUINDA = "#a51f4b"
ROSA = "#b85c6b"
FONDO = "#f6f5f5"

MESES_ES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]

def formato_miles(valor, _):
    return f"{int(valor):,}"


def etiqueta_mes(periodo):
    return f"{MESES_ES[periodo.month - 1]} {periodo.year}"


PERFIL = "reporte_diario"
cfg = Settings.desde_perfil(PERFIL)
u = Universo.construir(cfg, verbose=False)
FECHA_CORTE_RNPDNO = pd.Timestamp(u.meta["fecha_corte_rnpdno"]).normalize()
FECHA_CORTE_BANCI = pd.Timestamp(u.meta["fecha_corte_banci"]).normalize()
DIR_GRAFICAS = cfg.resultados_dir.parent / "Graficas_Referencias"
DIR_GRAFICAS.mkdir(parents=True, exist_ok=True)


## 1. Carpetas por mes

La serie mensual termina en el último mes con información verificada en BANCI. Los registros posteriores siguen en el universo completo, pero no se incluyen en las barras hasta su verificación.

In [ ]:
slice_carpetas = Slice("rn", cfg.periodo_match, estatus="des")
carpetas = u.slice(slice_carpetas).copy()

col_fecha = "fecha_hechos_modificada_derivada_rnpdno"
if col_fecha not in carpetas.columns:
    raise KeyError(f"El universo no contiene la fecha de período requerida: {col_fecha}")

carpetas["Mes"] = pd.to_datetime(carpetas[col_fecha], errors="coerce").dt.to_period("M")
if carpetas["Mes"].isna().any():
    raise ValueError("La rebanada de carpetas contiene fechas de desaparición modificadas nulas.")

carpetas["Particulares o forzada"] = (
    carpetas["is_cpar_derivada_rnpdno_banci"].astype(bool)
    | carpetas["is_dfor_derivada_rnpdno_banci"].astype(bool)
)
carpetas["Carpetas de investigación de otros delitos"] = (
    carpetas["is_cfval_derivada_rnpdno"].astype(bool)
    & ~carpetas["Particulares o forzada"]
)
inicio = pd.Period(pd.to_datetime(cfg.periodo_match.split("-")[0], format="%d%b%y"), freq="M")
fin = FECHA_CORTE_RNPDNO.to_period("M")
meses = pd.period_range(inicio, fin, freq="M")

nombre_particulares = "Carpetas por desaparición cometida por particulares o forzada"
nombre_otros = "Carpetas de investigación de otros delitos"
datos_mensuales = (
    carpetas.groupby("Mes", observed=True)[["Particulares o forzada", nombre_otros]]
    .sum()
    .rename(columns={"Particulares o forzada": nombre_particulares})
    .reindex(meses, fill_value=0)
    .astype(int)
)
datos_mensuales.index.name = "Mes"

totales_mensuales = datos_mensuales.sum(axis=1)
total_particulares = int(datos_mensuales[nombre_particulares].sum())
total_otros = int(datos_mensuales[nombre_otros].sum())
total_carpetas = int(totales_mensuales.sum())

mes_final_verificado = FECHA_CORTE_BANCI.to_period("M")
datos_grafica = datos_mensuales.loc[:mes_final_verificado].copy()
totales_graficados = datos_grafica.sum(axis=1)
total_carpetas_graficadas = int(totales_graficados.sum())

fig, ax = plt.subplots(figsize=(14.24, 5.92), dpi=100)
fig.patch.set_alpha(0)
ax.set_facecolor("none")
posiciones = np.arange(len(datos_grafica))
base = datos_grafica[nombre_particulares].to_numpy()
superior = datos_grafica[nombre_otros].to_numpy()

ax.bar(posiciones, base, width=0.9, color=GUINDA, label=nombre_particulares)
ax.bar(posiciones, superior, width=0.9, bottom=base, color=ROSA, label=nombre_otros)

for x, abajo, arriba, total in zip(posiciones, base, superior, totales_graficados):
    if abajo:
        ax.text(x, abajo / 2, formato_miles(abajo, None), ha="center", va="center", color="white", fontsize=12, fontweight="bold")
    if arriba:
        ax.text(x, abajo + arriba / 2, formato_miles(arriba, None), ha="center", va="center", color="white", fontsize=12, fontweight="bold")
    if total:
        ax.annotate(
            formato_miles(total, None), (x, total), xytext=(0, 5), textcoords="offset points",
            ha="center", va="bottom", color=GUINDA, fontsize=12,
        )

ax.set_xticks(posiciones, [etiqueta_mes(mes) for mes in datos_grafica.index], rotation=48, ha="right")
maximo = max(1, int(totales_graficados.max()))
ax.set_ylim(0, maximo * 1.22)
ax.yaxis.set_major_formatter(FuncFormatter(formato_miles))
ax.grid(axis="y", color="#d9d5d5", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color("#999999")
ax.tick_params(axis="y", length=0, labelsize=13.2)
ax.tick_params(axis="x", labelsize=13, length=0)


fig.tight_layout(rect=(0.02, 0.04, 0.98, 0.98))
fig.savefig(DIR_GRAFICAS / "carpetas_por_mes.svg", format="svg", transparent=True)
fig.savefig(DIR_GRAFICAS / "carpetas_por_mes.png", format="png", transparent=True)
display(fig)
plt.close(fig)

## 2. Datos faltantes en los registros reportados desde julio

La población es el conjunto dinámico que reproduce los 9,917 registros del marcador `dloc_v2` de la diapositiva 4. El desglose por campos usa los datos actuales del RNPDNO y conserva la institución del corte de julio cuando un registro ya no aparece en el RNPDNO vigente.

In [ ]:
rebanada_corte = Slice("rn", "01ene06-max", estatus="des", subfiltro="sisf")
cohorte_corte = bajas_vs_corte(
    u, rebanada_corte, "reg_rn_01jul26_des_sisf", sufijo="_v2", verbose=False
)
col_fub_corte = "fub_rnpdno_01jul26"
if col_fub_corte not in cohorte_corte.columns:
    raise KeyError(f"El corte no contiene la llave requerida: {col_fub_corte}")
if cohorte_corte[col_fub_corte].nunique() != len(cohorte_corte):
    raise ValueError("El corte de julio contiene registros duplicados por FUB.")

columnas_actuales = [
    "fub_rnpdno",
    "dependencia_origen_rnpdno",
    "idmunicipio_rnpdno",
    "fechanacimiento_rnpdno",
    "fechahechos_rnpdno",
    "nombre_vic_rnpdno",
    "primerapellido_vic_rnpdno",
    "segundoapellido_vic_rnpdno",
    "sexo_rnpdno",
    "curp_vic_rnpdno",
]
rnpdno_actual = pd.read_parquet(cfg.rnpdno_etl_parquet, columns=columnas_actuales)
fubs_actuales = rnpdno_actual["fub_rnpdno"].dropna()
if fubs_actuales.duplicated().any():
    raise ValueError("El RNPDNO actual contiene FUB duplicados; no se puede unir la cohorte sin ambigüedad.")

cohorte = cohorte_corte.merge(
    rnpdno_actual,
    how="left",
    left_on=col_fub_corte,
    right_on="fub_rnpdno",
    validate="one_to_one",
)
fuente_actual = cohorte["dependencia_origen_rnpdno"].astype("string")
sin_fuente_actual = fuente_actual.isna() | fuente_actual.str.strip().eq("").fillna(False)
fuente_corte = cohorte["dependencia_origen_derivada_rnpdno_banci_01jul26"].astype("string")
fuente = fuente_actual.where(~sin_fuente_actual, fuente_corte).fillna("")
fuente = fuente.map(
    lambda valor: unicodedata.normalize("NFKD", valor)
    .encode("ascii", "ignore").decode("ascii").upper().strip()
)
es_fiscalia = fuente.str.contains(r"FISCALIA|PROCURADUR", regex=True, na=False)
es_comision = fuente.str.contains(r"COMISION", regex=True, na=False)
cohorte["Institución"] = np.select(
    [es_fiscalia, es_comision], ["Fiscalías", "Comisiones"], default="Portal"
)

def esta_vacio(serie):
    texto = serie.astype("string").str.strip()
    return serie.isna() | texto.eq("").fillna(False)

nombre_vacio = pd.concat(
    [esta_vacio(cohorte[col]) for col in [
        "nombre_vic_rnpdno", "primerapellido_vic_rnpdno", "segundoapellido_vic_rnpdno"
    ]],
    axis=1,
).all(axis=1)
campos_faltantes = {
    "Municipio de\ndesaparición": esta_vacio(cohorte["idmunicipio_rnpdno"]),
    "Fecha de\nnacimiento": esta_vacio(cohorte["fechanacimiento_rnpdno"]),
    "Fecha de\ndesaparición": esta_vacio(cohorte["fechahechos_rnpdno"]),
    "Nombre": nombre_vacio,
    "Sexo": esta_vacio(cohorte["sexo_rnpdno"]),
    "CURP": esta_vacio(cohorte["curp_vic_rnpdno"]),
}
fuentes = ["Fiscalías", "Comisiones", "Portal"]
datos_faltantes = pd.DataFrame(
    {
        fuente: [int((mask & cohorte["Institución"].eq(fuente)).sum())
                 for mask in campos_faltantes.values()]
        for fuente in fuentes
    },
    index=list(campos_faltantes),
)
totales_faltantes = datos_faltantes.sum(axis=1)

colores_fuente = {"Fiscalías": "#418f9c", "Comisiones": "#e5ad3e", "Portal": "#e35e9b"}
fig, ax = plt.subplots(figsize=(16.34, 4.88), dpi=100)
fig.patch.set_alpha(0)
ax.set_facecolor("none")
posiciones = np.arange(len(datos_faltantes))
base = np.zeros(len(datos_faltantes), dtype=int)
etiquetas_exteriores = np.zeros(len(datos_faltantes), dtype=int)
for fuente in fuentes:
    valores = datos_faltantes[fuente].to_numpy()
    ax.bar(posiciones, valores, bottom=base, width=0.82, color=colores_fuente[fuente])
    for x, valor, inferior in zip(posiciones, valores, base):
        if valor >= 45:
            color_texto = "#222222" if fuente == "Comisiones" else "white"
            ax.text(x, inferior + valor / 2, formato_miles(valor, None),
                    ha="center", va="center", color=color_texto, fontsize=14, fontweight="bold")
        elif valor:
            ax.annotate(
                formato_miles(valor, None), (x, totales_faltantes.iloc[x]),
                xytext=(0, 4 + etiquetas_exteriores[x] * 20), textcoords="offset points",
                ha="center", va="bottom", color=colores_fuente[fuente], fontsize=14, fontweight="bold",
            )
            etiquetas_exteriores[x] += 1
    base += valores

ax.set_xticks(posiciones, list(campos_faltantes), rotation=0, ha="center", fontweight="bold")
maximo = max(1, int(totales_faltantes.max()))
ax.set_ylim(0, maximo * 1.28)
ax.yaxis.set_major_formatter(FuncFormatter(formato_miles))
ax.grid(axis="y", color="#d9d5d5", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color("#999999")
ax.tick_params(axis="y", length=0, labelsize=13.2)
ax.tick_params(axis="x", labelsize=13, length=0)
fig.tight_layout()
fig.savefig(DIR_GRAFICAS / "localizaciones_desde_corte_julio.svg", format="svg", transparent=True)
fig.savefig(DIR_GRAFICAS / "localizaciones_desde_corte_julio.png", format="png", transparent=True)
display(fig)
plt.close(fig)